# Problem 39: Scoped Memory

**Difficulty:** Medium | **Framework:** LangGraph

**Categories:** memory, orchestration

This notebook accompanies [Agent Foundry](https://agent-foundry-theta.vercel.app/problems/scoped-memory).

## 1. Install Dependencies

In [ ]:
!pip install -q langgraph langchain langchain-openai langchain-community

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Constraints

- Each agent must have its own private memory scope
- A shared scope must exist for cross-agent communication
- Private memory of agent A must never be visible to agent B
- The shared scope should only contain explicitly published information


## 4. Starter Code (has a bug — fix it!)

The code below has an intentional issue. Read the problem description and fix it.

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOpenAI(model="gpt-4o-mini")

# BUG: All agents share a single global memory — no scoping
global_memory = []

class State(TypedDict):
    task: str
    result: str

def researcher(state: State) -> State:
    global_memory.append("[Researcher internal] Raw data contains PII: user SSN 123-45-6789")
    global_memory.append("[Researcher] Found 3 relevant articles about AI safety.")
    context = "\n".join(global_memory)
    messages = [
        SystemMessage(content=f"You are a researcher. Memory:\n{context}"),
        HumanMessage(content=state["task"]),
    ]
    response = llm.invoke(messages)
    return {"result": response.content}

def writer(state: State) -> State:
    global_memory.append("[Writer internal] Draft has grammatical issues, fixing...")
    global_memory.append("[Writer] Completed first draft of the report.")
    # BUG: Writer can see researcher's private notes including PII
    context = "\n".join(global_memory)
    messages = [
        SystemMessage(content=f"You are a writer. Memory:\n{context}"),
        HumanMessage(content=f"Write a report based on: {state['result']}"),
    ]
    response = llm.invoke(messages)
    return {"result": response.content}

graph = StateGraph(State)
graph.add_node("researcher", researcher)
graph.add_node("writer", writer)
graph.add_edge(START, "researcher")
graph.add_edge("researcher", "writer")
graph.add_edge("writer", END)
app = graph.compile()

result = app.invoke({"task": "Research AI safety trends and write a summary."})
print(result["result"])
print(f"\nGlobal memory (visible to all): {global_memory}")

## 5. Your Solution

Modify the code above or write your fixed version below.

In [ ]:
# Write your solution here


## 6. Hints

1. Use a dictionary with namespace keys (e.g., 'agent_a_private', 'agent_b_private', 'shared') to separate memory scopes
2. Each agent should only read from its own private scope and the shared scope
3. Create helper functions like write_private(), write_shared(), and read_context() that enforce scope boundaries


## 7. Evaluation Criteria

Check your solution against these criteria:

- Private memory is isolated
- Shared scope enables collaboration
- Sensitive data stays private
